# Inteligência Artificial (PCO118A) — Aula 5
## Laboratório: Busca Heurística  ·  60 minutos
**Prof. Carlos Costa Cedro** · Bacharelado em Ciência da Computação · PUCPR

Complete as células marcadas com `TODO`. Execute na ordem.

### Como usar este notebook

São **cinco práticas** cronometradas. O tempo sugerido está no título de cada uma.
Se uma prática estourar o tempo, pule para a seguinte: elas são independentes, exceto
a Prática 2, que é usada por todas as outras.

| Tempo | Prática | O que você constrói |
|---|---|---|
| 5 min | Setup | Rodar a célula fornecida, sem escrever nada |
| 10 min | 1. Duas heurísticas | `h1`, `h2` e o teste de dominância |
| 15 min | 2. Uma busca, três algoritmos | A função que vira custo uniforme, gulosa ou A* trocando **uma linha** |
| 10 min | 3. O mapa que engana a gulosa | Reusar a mesma busca em outro problema |
| 10 min | 4. Quebrar a admissibilidade | Multiplicar `h` por 3 e ver o que acontece |
| 10 min | 5. Torneio de heurísticas | Conflito linear e a cadeia de dominância |

> **Regra do relatório, desde já:** toda medição reporta **duas colunas**, nós expandidos
> e custo da solução. Uma coluna sozinha não sustenta conclusão nenhuma.

---
## Setup — 5 minutos

Rode a célula abaixo e siga em frente. Nada aqui é para você escrever.

O tabuleiro do 8-puzzle é uma **tupla de 9 números**, lida de cima para baixo e da esquerda
para a direita, com `0` representando o espaço vazio. O objetivo é `(1,2,3,4,5,6,7,8,0)`.

A classe `Problema` é a formulação da Aula 3: estado inicial, sucessores e teste de objetivo.
Ela existe para que a mesma função de busca sirva ao 8-puzzle **e** ao mapa da Prática 3.

In [ ]:
# ============================================================
# CÓDIGO FORNECIDO — não altere
# ============================================================
import heapq, random, time

OBJETIVO = (1, 2, 3, 4, 5, 6, 7, 8, 0)

class Problema:
    """Formulação da Aula 3: inicial, sucessores(e) -> [(estado, custo)], e_objetivo(e)."""
    def __init__(self, inicial, sucessores, e_objetivo):
        self.inicial = inicial
        self.sucessores = sucessores
        self.e_objetivo = e_objetivo

def vizinhos_puzzle(e):
    """Move o espaço vazio nas quatro direções. Cada movimento custa 1."""
    z = e.index(0); r, c = divmod(z, 3); saida = []
    for dr, dc in ((-1,0), (1,0), (0,-1), (0,1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            k = nr*3 + nc
            l = list(e); l[z], l[k] = l[k], l[z]
            saida.append((tuple(l), 1))
    return saida

def puzzle(inicial):
    return Problema(inicial, vizinhos_puzzle, lambda e: e == OBJETIVO)

def mostrar(e):
    for r in range(3):
        print(' '.join('.' if v == 0 else str(v) for v in e[r*3:r*3+3]))

# As duas instâncias usadas em todo o laboratório
DIFICIL = (5, 3, 0, 1, 4, 6, 2, 7, 8)
MEDIA   = (1, 2, 0, 4, 8, 3, 5, 7, 6)

print('Instância DIFICIL:'); mostrar(DIFICIL)
print('\nInstância MEDIA:');  mostrar(MEDIA)
print('\nObjetivo:');         mostrar(OBJETIVO)

---
## Prática 1 — Duas heurísticas · 10 minutos

Vamos construir as duas heurísticas clássicas do 8-puzzle, ambas obtidas por **relaxamento**
do problema, como no slide 26.

**`h1` — peças fora do lugar.** Relaxamos as duas regras: qualquer peça voa para qualquer
casa. O custo mínimo passa a ser o número de peças que não estão no lugar certo.
O espaço vazio **não conta** como peça.

**`h2` — distância de Manhattan.** Relaxamos só a regra de a casa estar vazia: a peça anda
para qualquer casa vizinha, atravessando as outras. O custo mínimo é, para cada peça, a soma
das distâncias na vertical e na horizontal até seu lugar de destino.

> Dica para `h2`: a posição de destino da peça de valor `v` é `OBJETIVO.index(v)`. De um
> índice `i` você recupera linha e coluna com `divmod(i, 3)`.

In [ ]:
def h1(e):
    """Número de peças fora do lugar. Ignore o 0."""
    # TODO
    raise NotImplementedError('prática 1')

def h2(e):
    """Soma das distâncias de Manhattan de cada peça até seu destino. Ignore o 0."""
    # TODO
    raise NotImplementedError('prática 1')

In [ ]:
# Conferência (fornecida)
assert h1(OBJETIVO) == 0 and h2(OBJETIVO) == 0, 'no objetivo, nada falta'
assert h1(DIFICIL) == 7, f'h1(DIFICIL) deveria ser 7, deu {h1(DIFICIL)}'
assert h2(DIFICIL) == 10, f'h2(DIFICIL) deveria ser 10, deu {h2(DIFICIL)}'
assert h1(MEDIA) == 5 and h2(MEDIA) == 6, 'confira MEDIA também'

# Teste de dominância: h2 nunca pode ser menor que h1
rng = random.Random(1)
violacoes = 0
for _ in range(2000):
    e = tuple(rng.sample(range(9), 9))
    if h2(e) < h1(e):
        violacoes += 1
print(f'h1(DIFICIL) = {h1(DIFICIL)}   h2(DIFICIL) = {h2(DIFICIL)}')
print(f'violações de dominância em 2000 amostras: {violacoes}')
assert violacoes == 0, 'h2 deve dominar h1 em TODO estado'
print('\nPrática 1: OK — h2 domina h1')

**Para o relatório.** O teste acima verificou a dominância em 2000 amostras e não achou
violação. **Isso demonstra que h2 domina h1?** Justifique. Qual é a diferença entre esse
teste e o argumento do slide 26 sobre relaxamento?

---
## Prática 2 — Uma busca, três algoritmos · 15 minutos

Esta é a prática central. Você vai escrever **uma** função de busca que vira custo uniforme,
busca gulosa ou A* dependendo de um parâmetro.

```
prioridade = g          ->  custo uniforme
prioridade = h(estado)  ->  busca gulosa
prioridade = g + h(estado)  ->  A*
```

Dois detalhes que o slide 11 chamou de **decisões críticas**, e que valem a corretude inteira:

1. **O teste de objetivo acontece na expansão**, quando o nó sai da fila, nunca na geração.
   Testar na geração faz o código devolver soluções subótimas sem quebrar.
2. **Um estado já fechado é descartado.** Isso é seguro porque `h2` é consistente
   (Teorema 4). Com uma heurística inconsistente, seria um bug silencioso.

A função devolve `(custo_da_solucao, nos_expandidos)`.

In [ ]:
def busca(problema, modo, h=None):
    """modo: 'uniforme', 'gulosa' ou 'astar'. Devolve (custo, expandidos)."""
    if h is None:
        h = lambda e: 0

    def prioridade(g, estado):
        # TODO: devolver g, h(estado) ou g + h(estado) conforme o modo
        raise NotImplementedError('prática 2')

    inicial = problema.inicial
    fila = [(prioridade(0, inicial), 0, inicial)]
    melhor_g = {inicial: 0}
    fechados = set()
    expandidos = 0

    while fila:
        _, g, e = heapq.heappop(fila)
        if e in fechados:
            continue
        fechados.add(e)
        expandidos += 1
        # TODO: teste de objetivo AQUI, na expansão. Devolver (g, expandidos)
        raise NotImplementedError('prática 2')
        # TODO: para cada (v, custo) em problema.sucessores(e):
        #       novo_g = g + custo; pular se v já fechado ou se melhor_g[v] <= novo_g;
        #       registrar melhor_g[v] e empilhar (prioridade(novo_g, v), novo_g, v)
    return None, expandidos

In [ ]:
print(f"{'instância':<10}{'algoritmo':<18}{'nós expandidos':>16}{'custo':>8}")
resultados = {}
for nome, inst in (('DIFICIL', DIFICIL), ('MEDIA', MEDIA)):
    for rot, modo, hf in (('custo uniforme', 'uniforme', None),
                          ('gulosa (h2)',    'gulosa',   h2),
                          ('A* (h2)',        'astar',    h2)):
        t = time.time()
        custo, exp = busca(puzzle(inst), modo, hf)
        resultados[(nome, rot)] = (custo, exp)
        print(f'{nome:<10}{rot:<18}{exp:>16}{custo:>8}')
    print()

u = resultados[('DIFICIL', 'custo uniforme')][1]
a = resultados[('DIFICIL', 'A* (h2)')][1]
print(f'DIFICIL: o A* expandiu {u/a:.0f}x menos nós que o custo uniforme, com o mesmo custo.')

assert resultados[('DIFICIL', 'custo uniforme')] == (16, 8145)
assert resultados[('DIFICIL', 'A* (h2)')] == (16, 147)
assert resultados[('MEDIA', 'gulosa (h2)')] == (16, 40)
assert resultados[('MEDIA', 'A* (h2)')] == (12, 65)
print('\nPrática 2: OK')

**Para o relatório.**

1. Na instância **MEDIA**, a gulosa expandiu 40 nós e devolveu 16 movimentos; o A* expandiu
   65 e devolveu 12. A gulosa foi mais barata e entregou solução pior. **Em que situação
   essa troca é um bom negócio?**
2. Na instância **DIFICIL**, a gulosa acertou o custo ótimo. **Isso quer dizer que ela é
   ótima?** Cuidado com a resposta.
3. Troque `modo='gulosa'` por `modo='uniforme'` mentalmente: qual das duas colunas muda mais?

---
## Prática 3 — O mapa que engana a gulosa · 10 minutos

Este é o mapa do slide 8. Você **não escreve busca nenhuma aqui**: usa a mesma função da
Prática 2 em um problema completamente diferente. É para isso que a classe `Problema` existe.

```
          8            9
    O ---------> A ---------> D
    |                         ^
    | 5                       | 5
    v            4            |
    B ---------> C -----------+

    heurística (distância em linha reta até D):
    h(O)=10   h(A)=4   h(B)=8   h(C)=4   h(D)=0
```

Complete `problema_mapa()` construindo um `Problema` a partir do dicionário `MAPA`.

In [ ]:
MAPA = {'O': [('A', 8), ('B', 5)],
        'A': [('D', 9)],
        'B': [('C', 4)],
        'C': [('D', 5)],
        'D': []}
H_MAPA = {'O': 10, 'A': 4, 'B': 8, 'C': 4, 'D': 0}

def problema_mapa():
    # TODO: devolver Problema(inicial='O', sucessores=..., e_objetivo=...)
    raise NotImplementedError('prática 3')

def h_mapa(e):
    # TODO
    raise NotImplementedError('prática 3')

In [ ]:
cg, eg = busca(problema_mapa(), 'gulosa', h_mapa)
ca, ea = busca(problema_mapa(), 'astar',  h_mapa)
print(f'gulosa : custo {cg}, {eg} nós expandidos')
print(f'A*     : custo {ca}, {ea} nós expandidos')
print(f'\na gulosa pagou {cg - ca} a mais para expandir {ea - eg} nós a menos')

# A heurística é admissível e consistente? Vamos conferir em vez de acreditar.
def h_estrela(n):
    return 0 if n == 'D' else min(c + h_estrela(t) for t, c in MAPA[n])

admissivel  = all(H_MAPA[n] <= h_estrela(n) for n in MAPA)
consistente = all(H_MAPA[u] <= c + H_MAPA[v] for u in MAPA for v, c in MAPA[u])
print(f'h* = {{ {", ".join(f"{n}: {h_estrela(n)}" for n in MAPA)} }}')
print(f'admissível: {admissivel}    consistente: {consistente}')

assert (cg, ca) == (17, 14), 'gulosa deve pagar 17 e A* deve pagar 14'
assert admissivel and consistente
print('\nPrática 3: OK')

**Para o relatório.** A gulosa escolheu **A** porque `h(A)=4` é menor que `h(B)=8`, ou seja,
A parece mais perto do destino. **Quais duas informações ela ignorou** ao tomar essa decisão?
Aponte as duas usando os números do mapa.

---
## Prática 4 — Quebrar a admissibilidade · 10 minutos

O slide 12 diz que `h(n) <= h*(n)` é a condição que garante otimalidade. Vamos violá-la de
propósito e observar o estrago.

Defina `h_inadmissivel(e) = 3 * h2(e)`. Multiplicar por 3 faz a estimativa **superestimar**
o custo restante em muitos estados, o que é exatamente o que a admissibilidade proíbe.

Antes de rodar, **escreva sua previsão**: o algoritmo vai ficar mais rápido? A solução vai
piorar? Anote antes de ver o resultado.

In [ ]:
def h_inadmissivel(e):
    # TODO: 3 vezes a distância de Manhattan
    raise NotImplementedError('prática 4')

# TODO: mostrar, para DIFICIL e MEDIA, o resultado de A* com h2 e com h_inadmissivel,
#       imprimindo nós expandidos e custo da solução nos quatro casos.
raise NotImplementedError('prática 4')

**Para o relatório.** Compare as duas linhas com a heurística inadmissível:

- Na **DIFICIL**, ela ficou **mais rápida** (62 nós contra 147) e manteve o custo ótimo.
- Na **MEDIA**, ela ficou **mais lenta** (111 nós contra 65) **e** devolveu uma solução
  pior (16 movimentos em vez de 12).

1. Sua previsão bateu? Em qual das duas você errou?
2. Quebrar a admissibilidade garante que a busca fique mais rápida? Garante que a solução
   fique pior? **O que exatamente ela garante?**
3. Escreva em uma frase o que a admissibilidade compra, e o que se perde ao abrir mão dela.

---
## Prática 5 — Torneio de heurísticas · 10 minutos

Última prática. Vamos construir uma heurística que **domina** `h2` e medir o ganho.

**Conflito linear.** Duas peças estão em conflito quando estão ambas na sua linha de destino
e em **ordem invertida** entre si. Cada par nessa situação exige pelo menos **2 movimentos
além** da distância de Manhattan, porque uma das duas precisa sair da linha e voltar.
O mesmo vale para colunas.

`h3(e) = h2(e) + 2 * (número de pares em conflito linear)`

Como só somamos movimentos que são comprovadamente necessários, `h3` continua admissível,
e domina `h2` por construção.

In [ ]:
def h3(e):
    """Manhattan mais conflito linear."""
    extra = 0
    for r in range(3):
        # TODO: pegar as peças da linha r que JÁ estão na sua linha de destino,
        #       na ordem em que aparecem; listar a coluna de destino de cada uma;
        #       contar os pares fora de ordem e somar 2 por par
        pass
    for c in range(3):
        # TODO: o mesmo para colunas, usando a linha de destino
        pass
    raise NotImplementedError('prática 5')

In [ ]:
print(f"{'instância':<10}{'heurística':<26}{'nós':>8}{'custo':>8}")
torneio = {}
for nome, inst in (('DIFICIL', DIFICIL), ('MEDIA', MEDIA)):
    for rot, hf in (('h1 (peças fora)', h1),
                    ('h2 (Manhattan)', h2),
                    ('h3 (Manhattan + conflito)', h3)):
        custo, exp = busca(puzzle(inst), 'astar', hf)
        torneio[(nome, rot)] = exp
        print(f'{nome:<10}{rot:<26}{exp:>8}{custo:>8}')
    print()

# a cadeia de dominância se confirma ponto a ponto?
rng = random.Random(4)
amostras = [tuple(rng.sample(range(9), 9)) for _ in range(2000)]
assert all(h1(e) <= h2(e) <= h3(e) for e in amostras), 'a cadeia h1 <= h2 <= h3 deve valer'
print('cadeia h1 <= h2 <= h3 verificada em 2000 amostras')

assert torneio[('DIFICIL', 'h1 (peças fora)')] == 608
assert torneio[('DIFICIL', 'h2 (Manhattan)')] == 147
assert torneio[('DIFICIL', 'h3 (Manhattan + conflito)')] == 119
print('\nPrática 5: OK')

**Para o relatório.**

1. Na DIFICIL, a queda de `h1` para `h2` foi de 608 para 147 nós; de `h2` para `h3`, de 147
   para 119. **Por que o segundo ganho é tão menor que o primeiro?**
2. O Teorema 6 diz que a heurística que domina expande **no máximo** os mesmos nós. Ele diz
   alguma coisa sobre **tempo de execução**? Meça o tempo das três e responda com dados.
3. Se `h3` fosse dez vezes mais cara de calcular que `h2`, ainda valeria a pena? Escreva a
   conta que decidiria isso.

---
## Entrega

O relatório deve conter:

- respostas às perguntas das cinco práticas;
- **as duas colunas** em toda medição: nós expandidos e custo da solução. Relatórios com
  apenas uma das colunas serão devolvidos;
- o estado inicial usado em cada medição, para que o resultado seja reproduzível;
- uma frase dizendo se a sua implementação testa o objetivo na **expansão** ou na geração,
  e por que isso importa.

### Se sobrou tempo

- Meça o **fator de ramificação efetivo** b* de cada heurística, resolvendo
  `N + 1 = 1 + b* + (b*)^2 + ... + (b*)^d` numericamente por bisseção.
- Implemente o **A\* ponderado**, com `f = g + w*h` e `w = 1.5`, e verifique se o custo
  devolvido respeita o limite de `w` vezes o custo ótimo.